# Paneles solares: modelo final y test

El mejor detector de la validación cruzada fue **Faster R-CNN ResNet50 `partial`**: 0.331 de mAP50-95 macro, contra 0.271 del mejor YOLO (YOLO26n con el backbone congelado). Ganó en los tres folds.

Para el test se usa su receta con **anclas `paneles`, aumento `fuerte` y 20 épocas fijas**:

- En la validación cruzada empató con la base en mAP50-95 (0.332 contra 0.331), con mejor mAP50 (0.595 contra 0.566) y mejor detector como clasificador.
- Como no usa early stopping, no necesita una validación para elegir la época. Por eso se entrena **una sola vez con las 900 imágenes** y el modelo final es el de la última época. Los `best.pt` de cada fold vieron solo una parte de los datos.

Pasos:

1. Entrenar el modelo final con las 900 imágenes: unos 15 minutos en una T4. El resultado queda en `MyDrive/paneles_resultados/RCNN_final` y, si la sesión se corta, sigue desde la última época.
2. Evaluar en el test: 200 imágenes en 5 lotes, 125 con paneles y 75 sin, 178 cajas. Las etiquetas están en el repo (`data/test`). Las imágenes van al lado de cada etiqueta (`data/test/lote_N/test_XXX.png`), o en una carpeta de Drive que se indica más abajo.

En el test **no se elige nada**: ni época, ni umbral. El umbral del detector como clasificador es el de fábrica, 0.25, igual que en la validación cruzada.

In [1]:
import os

os.environ["PYTHONHASHSEED"] = "42"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Fuera de Colab: no se montó Drive.")

Mounted at /content/drive


In [2]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ana-castillo-zuain/procesamiento-de-imagenes.git"

if IN_COLAB:
    REPO_DIR = Path("/content/procesamiento-de-imagenes")
    if REPO_DIR.is_dir():
        # fetch + reset en vez de pull: un clon --depth 1 no siempre puede hacer fast-forward.
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    DRIVE = Path("/content/drive/MyDrive/paneles_resultados")
    RESULTS_DIR = DRIVE / "RCNN_final"
    # Si las imágenes del test no están en el repo, subilas a esta carpeta de Drive
    # (con o sin las subcarpetas lote_N): se ubican por nombre.
    TEST_IMAGES_DIR = DRIVE / "test_imagenes"
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "RCNN" else Path.cwd()
    RESULTS_DIR = REPO_DIR / "RCNN" / "results_final"
    TEST_IMAGES_DIR = None

CODE_DIR = REPO_DIR / "RCNN"
DATASET_DIR = REPO_DIR / "data" / "yolo"
TEST_DIR = REPO_DIR / "data" / "test"

assert CODE_DIR.joinpath("solar_rcnn", "final.py").is_file(), "El código clonado no tiene solar_rcnn/final.py."
assert DATASET_DIR.joinpath("PV01_val").is_dir(), f"No está el dataset en {DATASET_DIR}"
assert TEST_DIR.joinpath("etiquetas.csv").is_file(), f"No están las etiquetas del test en {TEST_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("código    :", CODE_DIR)
print("dataset   :", DATASET_DIR)
print("test      :", TEST_DIR)
print("resultados:", RESULTS_DIR)

código    : /content/procesamiento-de-imagenes/RCNN
dataset   : /content/procesamiento-de-imagenes/data/yolo
test      : /content/procesamiento-de-imagenes/data/test
resultados: /content/drive/MyDrive/paneles_resultados/RCNN_final


In [3]:
import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible. Activá una GPU T4 en el entorno de ejecución.")
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "torchvision", torchvision.__version__)

Tesla T4
torch 2.14.1+cu130 torchvision 0.29.1+cu130


In [4]:
import logging
import sys

sys.path.insert(0, str(CODE_DIR))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)

from solar_rcnn.final import (
    FINAL_ARCHITECTURE, FINAL_STRATEGY, all_training_records, attach_test_images,
    describe_test, evaluate_test, final_config, test_records, train_final,
)

config = final_config(DATASET_DIR, RESULTS_DIR)
print(f"Modelo: {FINAL_ARCHITECTURE} {FINAL_STRATEGY}, anclas {config.anchors}, aumento {config.augment}, "
      f"{config.epochs} épocas fijas, batch {config.batch}, lr {config.lr}.")
print("Imágenes de entrenamiento:", len(all_training_records(DATASET_DIR)))
print(describe_test(test_records(TEST_DIR, require_images=False)))

Modelo: fasterrcnn_resnet50_fpn partial, anclas paneles, aumento fuerte, 20 épocas fijas, batch 8, lr 0.01.
Imágenes de entrenamiento: 900
Test: 200 imágenes, 125 con paneles, 75 sin, 178 cajas.
  lote_1: 40 imágenes, 22 con paneles, 34 cajas
  lote_2: 40 imágenes, 26 con paneles, 31 cajas
  lote_3: 40 imágenes, 27 con paneles, 46 cajas
  lote_4: 40 imágenes, 26 con paneles, 36 cajas
  lote_5: 40 imágenes, 24 con paneles, 31 cajas


## 1. Entrenar con las 900 imágenes

Unos 15 minutos. Si `final.pt` ya existe en Drive, no se entrena de nuevo.

In [5]:
weights = train_final(config)
print("modelo final:", weights)

2026-11-09 23:29:23,669 CUDA disponible: Tesla T4


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 158MB/s]
2026-11-09 23:29:25,971 Modelo final fasterrcnn_resnet50_fpn partial, anclas paneles, aumento fuerte, 20 épocas con 900 imágenes. 41079331 parámetros entrenables.
/content/procesamiento-de-imagenes/RCNN/solar_rcnn/engine.py:728: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
2026-11-09 23:30:01,030 final época 1/20: loss 0.5459  (35 s)
2026-11-09 23:30:33,888 final época 2/20: loss 0.2670  (32 s)
2026-11-09 23:31:09,106 final época 3/20: loss 0.2399  (34 s)
2026-11-09 23:31:43,881 final época 4/20: loss 0.2212  (34 s)
2026-11-09 23:32:20,261 final época 5/20: loss 0.2082  (35 s)
20

modelo final: /content/drive/MyDrive/paneles_resultados/RCNN_final/final/weights/final.pt


## 2. Evaluar en el test

In [6]:
if TEST_IMAGES_DIR is not None and TEST_IMAGES_DIR.is_dir():
    print("imágenes copiadas desde Drive:", attach_test_images(TEST_DIR, TEST_IMAGES_DIR))

faltan = [r.path for r in test_records(TEST_DIR, require_images=False) if not r.path.is_file()]
if faltan:
    print(f"Faltan {len(faltan)} de 200 imágenes del test, por ejemplo {faltan[0].relative_to(TEST_DIR)}.")
    print(f"Subilas a {TEST_IMAGES_DIR} (o al repo, en data/test/lote_N/) y volvé a correr esta celda.")
else:
    resultado = evaluate_test(config, TEST_DIR, weights)
    print("test evaluado. Resultados en", RESULTS_DIR / "final" / "test")

2026-11-09 23:42:04,483 CUDA disponible: Tesla T4
2026-11-09 23:42:11,245 Métricas del test en /content/drive/MyDrive/paneles_resultados/RCNN_final/final/test


test evaluado. Resultados en /content/drive/MyDrive/paneles_resultados/RCNN_final/final/test


In [7]:
import pandas as pd

TEST_OUT = RESULTS_DIR / "final" / "test"
deteccion = pd.read_csv(TEST_OUT / "detection_metrics.csv")
clasificacion = pd.read_csv(TEST_OUT / "classification_metrics.csv")

display(deteccion.round(4))
display(clasificacion[["threshold_kind", "threshold", "tp", "fp", "fn", "tn", "accuracy", "precision", "recall", "f1", "f1_macro"]].round(4))

todo = deteccion.set_index("scope").loc["todo"]
print(f"Test: mAP50-95 {todo.map50_95:.3f}, mAP50 {todo.map50:.3f}, precisión {todo.precision:.3f}, recall {todo.recall:.3f}.")
print("Validación cruzada de la misma receta (promedio de 3 folds): mAP50-95 0.332, mAP50 0.595.")
print(f"Imágenes sin paneles con alguna detección >= 0.25: {int(todo.fp_images_in_negatives)} de {int(todo.n_negative_images)}.")

2026-11-09 23:42:11,602 NumExpr defaulting to 2 threads.


,scope,n_images,n_positive_images,n_negative_images,n_boxes,fp_images_in_negatives,fp_boxes_in_negatives,map50,map50_95,precision,recall,conf_at_best_f1
0,todo,200,125,75,178,50,96,0.7116,0.3511,0.6994,0.6798,0.9574
1,lote_1,40,22,18,34,10,25,0.7995,0.4121,0.8065,0.7353,0.9584
2,lote_2,40,26,14,31,10,19,0.8725,0.4827,0.8889,0.7742,0.9824
3,lote_3,40,27,13,46,12,26,0.5591,0.2869,0.5686,0.6304,0.8608
4,lote_4,40,26,14,36,8,13,0.6412,0.3246,0.6757,0.6944,0.9041
5,lote_5,40,24,16,31,10,13,0.7747,0.3374,0.7931,0.7419,0.9834


,threshold_kind,threshold,tp,fp,fn,tn,accuracy,precision,recall,f1,f1_macro
0,fabrica,0.25,123,50,2,25,0.740,0.711,0.984,0.8255,0.6578
1,trivial_todo_positivo,0.50,125,75,0,0,0.625,0.625,1.000,0.7692,0.3846


Test: mAP50-95 0.351, mAP50 0.712, precisión 0.699, recall 0.680.
Validación cruzada de la misma receta (promedio de 3 folds): mAP50-95 0.332, mAP50 0.595.
Imágenes sin paneles con alguna detección >= 0.25: 50 de 75.


## 3. Ejemplos

Verde: cajas reales. Rojo: detecciones con confianza ≥ 0.25, con su confianza.

In [8]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

registros = test_records(TEST_DIR)
puntajes = pd.read_csv(TEST_OUT / "predictions" / "image_scores.csv")
# Algunas imágenes con paneles y algunas sin, siempre las mismas.
elegidas = list(puntajes[puntajes.has_panels == 1].image[:5]) + list(puntajes[puntajes.has_panels == 0].image[:3])
por_nombre = {r.path.stem: r for r in registros}


def cajas_yolo(path, w, h, con_conf=False):
    cajas = []
    if path.is_file():
        for linea in path.read_text(encoding="utf-8").splitlines():
            v = [float(x) for x in linea.split()]
            if len(v) >= 5:
                cx, cy, bw, bh = v[1] * w, v[2] * h, v[3] * w, v[4] * h
                cajas.append((cx - bw / 2, cy - bh / 2, bw, bh, v[5] if con_conf and len(v) > 5 else None))
    return cajas


fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, nombre in zip(axes.flat, elegidas):
    r = por_nombre[nombre]
    imagen = Image.open(r.path).convert("RGB")
    w, h = imagen.size
    ax.imshow(imagen)
    for x, y, bw, bh, _ in cajas_yolo(r.label_path, w, h):
        ax.add_patch(patches.Rectangle((x, y), bw, bh, fill=False, color="lime", lw=1.5))
    for x, y, bw, bh, conf in cajas_yolo(TEST_OUT / "predictions" / "labels" / f"{nombre}.txt", w, h, True):
        ax.add_patch(patches.Rectangle((x, y), bw, bh, fill=False, color="red", lw=1.2))
        ax.text(x, y - 2, f"{conf:.2f}", color="red", fontsize=7)
    ax.set_title(f"{r.source}/{nombre}", fontsize=8)
    ax.axis("off")
fig.tight_layout()
plt.show()

---

Al terminar: **Archivo → Descargar → .ipynb**. En Drive quedan `RCNN_final/final/weights/final.pt` y, en `RCNN_final/final/test`, `detection_metrics.csv`, `classification_metrics.csv`, `metrics.json`, la curva PR y las predicciones.